# NuevaMente — Evaluación de Retrieval v1

## Objetivo

Este notebook evalúa la calidad del retrieval entregado por el equipo de Agentes usando el Ground Truth v1 de Data/IA.

Las métricas principales son:

- Recall@3
- Recall@5
- Precision@3
- Precision@5

## Alcance

Este notebook:

- consume resultados del retrieval;
- valida el contrato `retrieval_contract_v1`;
- compara los `chunk_id` recuperados contra `ground_truth_v1.csv`;
- calcula métricas de evaluación;
- analiza casos sin resultados y errores técnicos.

Este notebook **no implementa**:

- embeddings;
- Vector Store;
- retrieval productivo;
- generación de respuestas.

Esos componentes pertenecen al equipo de Agentes.

## 1. Configuración y carga de datos

En esta sección se configura la raíz del proyecto, se importan los módulos de Data/IA y se cargan el Ground Truth v1 y los archivos MOCK utilizados para validar el pipeline.

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

# ============================================================
# LOCALIZAR RAÍZ DEL PROYECTO
# ============================================================

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("data_ai existe:", (PROJECT_ROOT / "data_ai").exists())
print("PROJECT_ROOT en sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: C:\Users\MAMÁ\Downloads\NuevaMente
data_ai existe: True
PROJECT_ROOT en sys.path: True


In [2]:
from data_ai.metrics.retrieval_metrics import (
    recall_at_k,
    precision_at_k,
)

from data_ai.validators.retrieval_validator import (
    validate_retrieval_contract,
)

from data_ai.metrics.retrieval_evaluator import (
    parse_relevant_chunk_ids,
    extract_retrieved_chunk_ids,
    evaluate_retrieval_case,
    evaluate_retrieval_batch,
)

from data_ai.metrics.retrieval_reporting import (
    get_case_result,
    build_metric_summary,
    build_status_summary,
    build_category_summary,
    print_batch_report,
)

from data_ai.loaders.retrieval_loader import (
    load_retrieval_batch,
)

print("✅ Módulos de data_ai importados correctamente.")

✅ Módulos de data_ai importados correctamente.


In [3]:
# ============================================================
# RUTAS
# ============================================================

DATA_DIR = PROJECT_ROOT / "data"
EVALUATION_DIR = DATA_DIR / "evaluation"
MOCK_DIR = EVALUATION_DIR / "mock"

GROUND_TRUTH_PATH = EVALUATION_DIR / "ground_truth_v2.csv"

SUCCESS_MOCK_PATH = MOCK_DIR / "retrieval_success_v1.json"
NO_RESULTS_MOCK_PATH = MOCK_DIR / "retrieval_no_results_v1.json"
ERROR_MOCK_PATH = MOCK_DIR / "retrieval_error_v1.json"

TOP_K = 5

In [4]:
required_paths = [
    GROUND_TRUTH_PATH,
    SUCCESS_MOCK_PATH,
    NO_RESULTS_MOCK_PATH,
    ERROR_MOCK_PATH,
]

for path in required_paths:
    print("✅" if path.exists() else "❌", path)

✅ C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\ground_truth_v2.csv
✅ C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\mock\retrieval_success_v1.json
✅ C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\mock\retrieval_no_results_v1.json
✅ C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\mock\retrieval_error_v1.json


In [5]:
ground_truth_df = pd.read_csv(
    GROUND_TRUTH_PATH,
    encoding="utf-8-sig"
)

print("Filas:", len(ground_truth_df))
print("Casos únicos:", ground_truth_df["case_id"].nunique())

ground_truth_df.head()

Filas: 50
Casos únicos: 50


,case_id,document_id,categoria,pregunta,respuesta_esperada,seccion_evidencia,palabras_clave_evidencia,dificultad,estado_prueba,retrieved_chunk_ids,relevant_retrieved,notas,relevant_chunk_ids,num_relevant_chunks,ground_truth_status
0,AI-ES-001-Q01,AI-ES-001,IA,¿Qué pretende introducir este módulo sobre int...,Introduce los tipos de soluciones que puede ha...,Objetivos de aprendizaje,soluciones de IA; inteligencia artificial resp...,Básica,Pendiente de ejecutar,NaN,NaN,NaN,AI-ES-001_CH_001,1,Validado
1,AI-ES-001-Q02,AI-ES-001,IA,¿Es obligatorio conocer aprendizaje automático...,No. El conocimiento conceptual de aprendizaje ...,Requisitos previos,no obligatorio; aprendizaje automático; útil,Básica,Pendiente de ejecutar,NaN,NaN,NaN,AI-ES-001_CH_001,1,Validado
2,AI-ES-001-Q03,AI-ES-001,IA,Menciona dos cargas de trabajo de IA tratadas ...,"Por ejemplo: texto y lenguaje natural, discurs...",Unidades del módulo,texto; lenguaje natural; discurso; visión por ...,Básica,Pendiente de ejecutar,NaN,NaN,NaN,AI-ES-001_CH_002,1,Validado
3,AI-ES-001-Q04,AI-ES-001,IA,¿Qué tema del módulo aborda el uso ético o seg...,La inteligencia artificial responsable.,Unidades del módulo,inteligencia artificial responsable,Básica,Pendiente de ejecutar,NaN,NaN,NaN,AI-ES-001_CH_002,1,Validado
4,AI-ES-001-Q05,AI-ES-001,IA,¿Qué conocimiento previo puede ser útil antes ...,Un conocimiento conceptual del aprendizaje aut...,Requisitos previos,aprendizaje automático; conocimiento conceptua...,Básica,Pendiente de ejecutar,NaN,NaN,Pregunta reemplazada porque la pregunta origin...,AI-ES-001_CH_001,1,Validado


In [6]:
assert len(ground_truth_df) == 50, \
    "El Ground Truth debe contener 50 casos."

assert ground_truth_df["case_id"].nunique() == 50, \
    "Los case_id deben ser únicos."

assert ground_truth_df["case_id"].notna().all(), \
    "Hay case_id vacíos."

assert ground_truth_df["relevant_chunk_ids"].notna().all(), \
    "Hay casos sin relevant_chunk_ids."

print("✅ Ground Truth v1 validado correctamente.")

✅ Ground Truth v1 validado correctamente.


In [7]:
def load_json(path: Path) -> dict:
    """Carga un archivo JSON y devuelve su contenido como diccionario."""
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

In [8]:
success_mock = load_json(SUCCESS_MOCK_PATH)
no_results_mock = load_json(NO_RESULTS_MOCK_PATH)
error_mock = load_json(ERROR_MOCK_PATH)

print("Success status:", success_mock["status"])
print("No results status:", no_results_mock["status"])
print("Error status:", error_mock["status"])

Success status: success
No results status: no_results
Error status: error


## 2. Validación del contrato de retrieval

Antes de calcular métricas, Data/IA valida que la respuesta entregada por Agentes cumpla el contrato `retrieval_contract_v1`.

Se validan:

- versión del contrato;
- `case_id`;
- `query`;
- `top_k`;
- `score_type`;
- `status`;
- estructura de `results`;
- estructura de `error`;
- consistencia entre `status`, `results` y `error`.

In [9]:
def print_contract_validation(
    name: str,
    payload: dict,
) -> None:
    """
    Ejecuta la validación y muestra un resumen legible.
    """

    errors = validate_retrieval_contract(payload)

    if not errors:
        print(
            f"✅ {name}: contrato válido "
            f"(status={payload['status']})"
        )
        return

    print(
        f"❌ {name}: se encontraron "
        f"{len(errors)} error(es)"
    )

    for error in errors:
        print(
            f"   - {error}"
        )

In [10]:
print_contract_validation(
    "retrieval_success_v1",
    success_mock,
)

print_contract_validation(
    "retrieval_no_results_v1",
    no_results_mock,
)

print_contract_validation(
    "retrieval_error_v1",
    error_mock,
)

✅ retrieval_success_v1: contrato válido (status=success)
✅ retrieval_no_results_v1: contrato válido (status=no_results)
✅ retrieval_error_v1: contrato válido (status=error)


In [11]:
from copy import deepcopy

invalid_mock = deepcopy(success_mock)

invalid_mock["status"] = "success"
invalid_mock["results"] = []
invalid_mock["error"] = None

print_contract_validation(
    "invalid_mock",
    invalid_mock,
)

❌ invalid_mock: se encontraron 1 error(es)
   - Value error, status='success' requiere al menos un resultado.


## 3. Preparación de resultados para métricas

Para calcular Recall@K y Precision@K necesitamos extraer, en orden de ranking, los `chunk_id` recuperados por el retrieval.

La evaluación compara:

- `retrieved_chunk_ids`: chunks recuperados por Agentes;
- `relevant_chunk_ids`: chunks definidos como relevantes en el Ground Truth.

Para `status = no_results` o `status = error`, la lista recuperada será vacía.

In [12]:
success_ids = extract_retrieved_chunk_ids(success_mock)
no_results_ids = extract_retrieved_chunk_ids(no_results_mock)
error_ids = extract_retrieved_chunk_ids(error_mock)

print("success:", success_ids)
print("no_results:", no_results_ids)
print("error:", error_ids)

success: ['CLD-ES-001_CH_001', 'CLD-ES-001_CH_002', 'CLD-ES-001_CH_003']
no_results: []
error: []


In [13]:
ground_truth_df[
    [
        "case_id",
        "pregunta",
        "relevant_chunk_ids",
    ]
].head(10)

,case_id,pregunta,relevant_chunk_ids
0,AI-ES-001-Q01,¿Qué pretende introducir este módulo sobre int...,AI-ES-001_CH_001
1,AI-ES-001-Q02,¿Es obligatorio conocer aprendizaje automático...,AI-ES-001_CH_001
2,AI-ES-001-Q03,Menciona dos cargas de trabajo de IA tratadas ...,AI-ES-001_CH_002
3,AI-ES-001-Q04,¿Qué tema del módulo aborda el uso ético o seg...,AI-ES-001_CH_002
4,AI-ES-001-Q05,¿Qué conocimiento previo puede ser útil antes ...,AI-ES-001_CH_001
5,AI-ES-002-Q01,¿Qué conceptos principales de IA generativa ab...,AI-ES-002_CH_001
6,AI-ES-002-Q02,¿Qué debe poder explicar el estudiante sobre l...,AI-ES-002_CH_001
7,AI-ES-002-Q03,¿Qué habilidad relacionada con prompts se espe...,AI-ES-002_CH_001
8,AI-ES-002-Q04,¿Qué conocimientos previos recomienda el módulo?,AI-ES-002_CH_002
9,AI-ES-002-Q05,¿El módulo incluye agentes de IA?,AI-ES-002_CH_001;AI-ES-002_CH_002


In [14]:
print(
    ground_truth_df[
        ["case_id", "relevant_chunk_ids"]
    ].head(10).to_string(index=False)
)

      case_id                relevant_chunk_ids
AI-ES-001-Q01                  AI-ES-001_CH_001
AI-ES-001-Q02                  AI-ES-001_CH_001
AI-ES-001-Q03                  AI-ES-001_CH_002
AI-ES-001-Q04                  AI-ES-001_CH_002
AI-ES-001-Q05                  AI-ES-001_CH_001
AI-ES-002-Q01                  AI-ES-002_CH_001
AI-ES-002-Q02                  AI-ES-002_CH_001
AI-ES-002-Q03                  AI-ES-002_CH_001
AI-ES-002-Q04                  AI-ES-002_CH_002
AI-ES-002-Q05 AI-ES-002_CH_001;AI-ES-002_CH_002


In [15]:
ground_truth_df["relevant_chunk_ids_list"] = (
    ground_truth_df["relevant_chunk_ids"]
    .apply(parse_relevant_chunk_ids)
)

ground_truth_df[
    [
        "case_id",
        "relevant_chunk_ids",
        "relevant_chunk_ids_list",
    ]
].head(10)

,case_id,relevant_chunk_ids,relevant_chunk_ids_list
0,AI-ES-001-Q01,AI-ES-001_CH_001,[AI-ES-001_CH_001]
1,AI-ES-001-Q02,AI-ES-001_CH_001,[AI-ES-001_CH_001]
2,AI-ES-001-Q03,AI-ES-001_CH_002,[AI-ES-001_CH_002]
3,AI-ES-001-Q04,AI-ES-001_CH_002,[AI-ES-001_CH_002]
4,AI-ES-001-Q05,AI-ES-001_CH_001,[AI-ES-001_CH_001]
5,AI-ES-002-Q01,AI-ES-002_CH_001,[AI-ES-002_CH_001]
6,AI-ES-002-Q02,AI-ES-002_CH_001,[AI-ES-002_CH_001]
7,AI-ES-002-Q03,AI-ES-002_CH_001,[AI-ES-002_CH_001]
8,AI-ES-002-Q04,AI-ES-002_CH_002,[AI-ES-002_CH_002]
9,AI-ES-002-Q05,AI-ES-002_CH_001;AI-ES-002_CH_002,"[AI-ES-002_CH_001, AI-ES-002_CH_002]"


In [16]:
mock_case_id = success_mock["case_id"]

mock_ground_truth = ground_truth_df.loc[
    ground_truth_df["case_id"] == mock_case_id
]

mock_ground_truth[
    [
        "case_id",
        "pregunta",
        "relevant_chunk_ids_list",
    ]
]

,case_id,pregunta,relevant_chunk_ids_list
40,CLD-ES-001-Q01,¿Qué es Kubernetes?,[CLD-ES-001_CH_001]


In [17]:
relevant_ids = mock_ground_truth.iloc[0][
    "relevant_chunk_ids_list"
]

retrieved_ids = extract_retrieved_chunk_ids(
    success_mock
)

print("Relevant:")
print(relevant_ids)

print("\nRetrieved:")
print(retrieved_ids)

Relevant:
['CLD-ES-001_CH_001']

Retrieved:
['CLD-ES-001_CH_001', 'CLD-ES-001_CH_002', 'CLD-ES-001_CH_003']


In [18]:
recall_3 = recall_at_k(
    relevant_ids,
    retrieved_ids,
    k=3,
)

recall_5 = recall_at_k(
    relevant_ids,
    retrieved_ids,
    k=5,
)

precision_3 = precision_at_k(
    relevant_ids,
    retrieved_ids,
    k=3,
)

precision_5 = precision_at_k(
    relevant_ids,
    retrieved_ids,
    k=5,
)

print(f"Recall@3:    {recall_3:.4f}")
print(f"Recall@5:    {recall_5:.4f}")
print(f"Precision@3: {precision_3:.4f}")
print(f"Precision@5: {precision_5:.4f}")

Recall@3:    1.0000
Recall@5:    1.0000
Precision@3: 0.3333
Precision@5: 0.2000


## 4. Evaluación individual de un caso

Se implementa una función reutilizable que:

1. valida el contrato del retrieval;
2. localiza el caso correspondiente en el Ground Truth;
3. obtiene los chunks relevantes;
4. obtiene los chunks recuperados;
5. calcula Recall@3, Recall@5, Precision@3 y Precision@5;
6. conserva información sobre el estado del retrieval.

Esto permitirá aplicar posteriormente el mismo proceso a los 50 casos.

In [19]:
success_evaluation = evaluate_retrieval_case(
    success_mock,
    ground_truth_df,
)

success_evaluation

{'case_id': 'CLD-ES-001-Q01',
 'categoria': 'Cloud / DevOps',
 'pregunta': '¿Qué es Kubernetes?',
 'status': 'success',
 'metric_eligible': True,
 'relevant_chunk_ids': ['CLD-ES-001_CH_001'],
 'retrieved_chunk_ids': ['CLD-ES-001_CH_001',
  'CLD-ES-001_CH_002',
  'CLD-ES-001_CH_003'],
 'recall_at_3': 1.0,
 'recall_at_5': 1.0,
 'precision_at_3': 0.3333333333333333,
 'precision_at_5': 0.2,
 'error_code': None,
 'error_message': None}

In [20]:
no_results_evaluation = evaluate_retrieval_case(
    no_results_mock,
    ground_truth_df,
)

no_results_evaluation

{'case_id': 'CLD-ES-001-Q01',
 'categoria': 'Cloud / DevOps',
 'pregunta': '¿Qué es Kubernetes?',
 'status': 'no_results',
 'metric_eligible': True,
 'relevant_chunk_ids': ['CLD-ES-001_CH_001'],
 'retrieved_chunk_ids': [],
 'recall_at_3': 0.0,
 'recall_at_5': 0.0,
 'precision_at_3': 0.0,
 'precision_at_5': 0.0,
 'error_code': None,
 'error_message': None}

In [21]:
error_evaluation = evaluate_retrieval_case(
    error_mock,
    ground_truth_df,
)

error_evaluation

{'case_id': 'CLD-ES-001-Q01',
 'categoria': 'Cloud / DevOps',
 'pregunta': '¿Qué es Kubernetes?',
 'status': 'error',
 'metric_eligible': False,
 'relevant_chunk_ids': ['CLD-ES-001_CH_001'],
 'retrieved_chunk_ids': [],
 'recall_at_3': None,
 'recall_at_5': None,
 'precision_at_3': None,
 'precision_at_5': None,
 'error_code': 'RETRIEVAL_FAILED',
 'error_message': 'No fue posible ejecutar la búsqueda vectorial.'}

## 5. Tabla estructurada de evaluación

Los resultados individuales se consolidan en un DataFrame para permitir:

- evaluación por lote;
- cálculo de promedios globales;
- análisis por categoría;
- identificación de casos fallidos;
- separación entre fallos técnicos y errores de retrieval.

In [22]:
mock_evaluations = [
    success_evaluation,
    no_results_evaluation,
    error_evaluation,
]

mock_results_df = pd.DataFrame(
    mock_evaluations
)

mock_results_df

,case_id,categoria,pregunta,status,metric_eligible,relevant_chunk_ids,retrieved_chunk_ids,recall_at_3,recall_at_5,precision_at_3,precision_at_5,error_code,error_message
0,CLD-ES-001-Q01,Cloud / DevOps,¿Qué es Kubernetes?,success,True,[CLD-ES-001_CH_001],"[CLD-ES-001_CH_001, CLD-ES-001_CH_002, CLD-ES-...",1.0,1.0,0.333333,0.2,None,None
1,CLD-ES-001-Q01,Cloud / DevOps,¿Qué es Kubernetes?,no_results,True,[CLD-ES-001_CH_001],[],0.0,0.0,0.000000,0.0,None,None
2,CLD-ES-001-Q01,Cloud / DevOps,¿Qué es Kubernetes?,error,False,[CLD-ES-001_CH_001],[],NaN,NaN,NaN,NaN,RETRIEVAL_FAILED,No fue posible ejecutar la búsqueda vectorial.


In [23]:
mock_results_df[
    [
        "case_id",
        "status",
        "metric_eligible",
        "recall_at_3",
        "recall_at_5",
        "precision_at_3",
        "precision_at_5",
        "error_code",
    ]
]

,case_id,status,metric_eligible,recall_at_3,recall_at_5,precision_at_3,precision_at_5,error_code
0,CLD-ES-001-Q01,success,True,1.0,1.0,0.333333,0.2,None
1,CLD-ES-001-Q01,no_results,True,0.0,0.0,0.000000,0.0,None
2,CLD-ES-001-Q01,error,False,NaN,NaN,NaN,NaN,RETRIEVAL_FAILED


In [24]:
metric_results_df = mock_results_df.loc[
    mock_results_df["metric_eligible"]
].copy()

technical_errors_df = mock_results_df.loc[
    ~mock_results_df["metric_eligible"]
].copy()

print(
    "Casos evaluables:",
    len(metric_results_df)
)

print(
    "Errores técnicos:",
    len(technical_errors_df)
)

Casos evaluables: 2
Errores técnicos: 1


In [25]:
metric_columns = [
    "recall_at_3",
    "recall_at_5",
    "precision_at_3",
    "precision_at_5",
]

mock_metric_summary = (
    metric_results_df[metric_columns]
    .mean()
    .to_frame(name="mean")
)

mock_metric_summary

,mean
recall_at_3,0.500000
recall_at_5,0.500000
precision_at_3,0.166667
precision_at_5,0.100000


## 6. Evaluación por lote

El objetivo de esta sección es evaluar múltiples respuestas de retrieval de forma automática.

El proceso debe:

1. validar cada payload;
2. detectar `case_id` duplicados;
3. detectar casos faltantes;
4. detectar casos extra no presentes en Ground Truth;
5. evaluar casos válidos;
6. separar errores técnicos;
7. devolver un DataFrame consolidado.

In [26]:
mock_payloads = [
    success_mock,
    no_results_mock,
    error_mock,
]

batch_results_df, batch_report = (
    evaluate_retrieval_batch(
        mock_payloads,
        ground_truth_df,
    )
)

In [27]:
batch_report

{'expected_cases': 50,
 'received_payloads': 3,
 'evaluated_cases': 3,
 'missing_case_ids': ['AI-ES-001-Q01',
  'AI-ES-001-Q02',
  'AI-ES-001-Q03',
  'AI-ES-001-Q04',
  'AI-ES-001-Q05',
  'AI-ES-002-Q01',
  'AI-ES-002-Q02',
  'AI-ES-002-Q03',
  'AI-ES-002-Q04',
  'AI-ES-002-Q05',
  'BE-ES-001-Q01',
  'BE-ES-001-Q02',
  'BE-ES-001-Q03',
  'BE-ES-001-Q04',
  'BE-ES-001-Q05',
  'BE-ES-002-Q01',
  'BE-ES-002-Q02',
  'BE-ES-002-Q03',
  'BE-ES-002-Q04',
  'BE-ES-002-Q05',
  'CLD-ES-001-Q02',
  'CLD-ES-001-Q03',
  'CLD-ES-001-Q04',
  'CLD-ES-001-Q05',
  'CLD-ES-002-Q01',
  'CLD-ES-002-Q02',
  'CLD-ES-002-Q03',
  'CLD-ES-002-Q04',
  'CLD-ES-002-Q05',
  'DS-ES-001-Q01',
  'DS-ES-001-Q02',
  'DS-ES-001-Q03',
  'DS-ES-001-Q04',
  'DS-ES-001-Q05',
  'DS-ES-002-Q01',
  'DS-ES-002-Q02',
  'DS-ES-002-Q03',
  'DS-ES-002-Q04',
  'DS-ES-002-Q05',
  'FE-ES-001-Q01',
  'FE-ES-001-Q02',
  'FE-ES-001-Q03',
  'FE-ES-001-Q04',
  'FE-ES-001-Q05',
  'FE-ES-002-Q01',
  'FE-ES-002-Q02',
  'FE-ES-002-Q03',
  'FE-E

In [28]:
batch_results_df[
    [
        "case_id",
        "status",
        "metric_eligible",
        "recall_at_3",
        "recall_at_5",
        "precision_at_3",
        "precision_at_5",
    ]
]

,case_id,status,metric_eligible,recall_at_3,recall_at_5,precision_at_3,precision_at_5
0,CLD-ES-001-Q01,success,True,1.0,1.0,0.333333,0.2
1,CLD-ES-001-Q01,no_results,True,0.0,0.0,0.000000,0.0
2,CLD-ES-001-Q01,error,False,NaN,NaN,NaN,NaN


In [29]:
print_batch_report(
    batch_report
)

Casos esperados: 50
Payloads recibidos: 3
Casos evaluados: 3
Casos faltantes: 49
Casos extra: 0
Case ID duplicados: 1
Errores de procesamiento: 0


## 7. Lote de prueba controlado

Se construye un lote pequeño con distintos escenarios para verificar el comportamiento de las métricas antes de utilizar resultados reales de Agentes.

Escenarios incluidos:

1. evidencia relevante en rank 1;
2. evidencia relevante fuera del Top-3 pero dentro del Top-5;
3. caso con dos chunks relevantes;
4. `no_results`;
5. error técnico.

Estos resultados son simulados y se utilizan únicamente para validar el pipeline de evaluación.

In [30]:
def build_mock_result(
    rank: int,
    chunk_id: str,
    document_id: str,
    score: float,
) -> dict:
    """
    Construye un resultado de retrieval simulado.
    """

    return {
        "rank": rank,
        "chunk_id": chunk_id,
        "document_id": document_id,
        "score": score,
        "text": f"Texto simulado para {chunk_id}",
        "metadata": {},
    }

In [31]:
mock_case_1 = {
    "contract_version": "1.0",
    "case_id": "AI-ES-001-Q01",
    "query": ground_truth_df.loc[
        ground_truth_df["case_id"] == "AI-ES-001-Q01",
        "pregunta",
    ].iloc[0],
    "top_k": 5,
    "score_type": "cosine_similarity",
    "status": "success",
    "results": [
        build_mock_result(
            1,
            "AI-ES-001_CH_001",
            "AI-ES-001",
            0.95,
        ),
        build_mock_result(
            2,
            "AI-ES-001_CH_002",
            "AI-ES-001",
            0.88,
        ),
        build_mock_result(
            3,
            "AI-ES-002_CH_001",
            "AI-ES-002",
            0.80,
        ),
        build_mock_result(
            4,
            "BE-ES-001_CH_001",
            "BE-ES-001",
            0.72,
        ),
        build_mock_result(
            5,
            "DS-ES-001_CH_001",
            "DS-ES-001",
            0.65,
        ),
    ],
    "error": None,
}

In [32]:
mock_case_2 = {
    "contract_version": "1.0",
    "case_id": "AI-ES-001-Q03",
    "query": ground_truth_df.loc[
        ground_truth_df["case_id"] == "AI-ES-001-Q03",
        "pregunta",
    ].iloc[0],
    "top_k": 5,
    "score_type": "cosine_similarity",
    "status": "success",
    "results": [
        build_mock_result(
            1,
            "AI-ES-001_CH_001",
            "AI-ES-001",
            0.92,
        ),
        build_mock_result(
            2,
            "AI-ES-002_CH_001",
            "AI-ES-002",
            0.86,
        ),
        build_mock_result(
            3,
            "BE-ES-001_CH_001",
            "BE-ES-001",
            0.81,
        ),
        build_mock_result(
            4,
            "AI-ES-001_CH_002",
            "AI-ES-001",
            0.76,
        ),
        build_mock_result(
            5,
            "DS-ES-001_CH_001",
            "DS-ES-001",
            0.70,
        ),
    ],
    "error": None,
}

In [33]:
mock_case_3 = {
    "contract_version": "1.0",
    "case_id": "AI-ES-002-Q05",
    "query": ground_truth_df.loc[
        ground_truth_df["case_id"] == "AI-ES-002-Q05",
        "pregunta",
    ].iloc[0],
    "top_k": 5,
    "score_type": "cosine_similarity",
    "status": "success",
    "results": [
        build_mock_result(
            1,
            "AI-ES-002_CH_001",
            "AI-ES-002",
            0.94,
        ),
        build_mock_result(
            2,
            "BE-ES-001_CH_001",
            "BE-ES-001",
            0.87,
        ),
        build_mock_result(
            3,
            "DS-ES-001_CH_001",
            "DS-ES-001",
            0.79,
        ),
        build_mock_result(
            4,
            "AI-ES-002_CH_002",
            "AI-ES-002",
            0.73,
        ),
        build_mock_result(
            5,
            "FE-ES-001_CH_001",
            "FE-ES-001",
            0.68,
        ),
    ],
    "error": None,
}

In [34]:
mock_case_4 = {
    "contract_version": "1.0",
    "case_id": "BE-ES-001-Q01",
    "query": ground_truth_df.loc[
        ground_truth_df["case_id"] == "BE-ES-001-Q01",
        "pregunta",
    ].iloc[0],
    "top_k": 5,
    "score_type": "cosine_similarity",
    "status": "no_results",
    "results": [],
    "error": None,
}

In [35]:
mock_case_5 = {
    "contract_version": "1.0",
    "case_id": "DS-ES-001-Q01",
    "query": ground_truth_df.loc[
        ground_truth_df["case_id"] == "DS-ES-001-Q01",
        "pregunta",
    ].iloc[0],
    "top_k": 5,
    "score_type": "cosine_similarity",
    "status": "error",
    "results": [],
    "error": {
        "code": "RETRIEVAL_FAILED",
        "message": "Error técnico simulado.",
    },
}

In [36]:
controlled_mock_batch = [
    mock_case_1,
    mock_case_2,
    mock_case_3,
    mock_case_4,
    mock_case_5,
]

In [37]:
controlled_results_df, controlled_report = (
    evaluate_retrieval_batch(
        controlled_mock_batch,
        ground_truth_df,
    )
)

In [38]:
controlled_results_df[
    [
        "case_id",
        "status",
        "metric_eligible",
        "recall_at_3",
        "recall_at_5",
        "precision_at_3",
        "precision_at_5",
        "error_code",
    ]
]

,case_id,status,metric_eligible,recall_at_3,recall_at_5,precision_at_3,precision_at_5,error_code
0,AI-ES-001-Q01,success,True,1.0,1.0,0.333333,0.2,None
1,AI-ES-001-Q03,success,True,0.0,1.0,0.000000,0.2,None
2,AI-ES-002-Q05,success,True,0.5,1.0,0.333333,0.4,None
3,BE-ES-001-Q01,no_results,True,0.0,0.0,0.000000,0.0,None
4,DS-ES-001-Q01,error,False,NaN,NaN,NaN,NaN,RETRIEVAL_FAILED


In [39]:
print_batch_report(
    controlled_report
)

Casos esperados: 50
Payloads recibidos: 5
Casos evaluados: 5
Casos faltantes: 45
Casos extra: 0
Case ID duplicados: 0
Errores de procesamiento: 0


## 8. Validación automática del pipeline

Antes de conectar resultados reales del equipo de Agentes, se verifican automáticamente escenarios controlados.

Estas pruebas permiten comprobar que:

- Recall@3 y Recall@5 reaccionan correctamente al ranking;
- se maneja evidencia múltiple;
- `no_results` se evalúa como recuperación fallida;
- los errores técnicos quedan fuera de las métricas.

In [40]:
# ------------------------------------------------------------
# Caso 1 — evidencia relevante en rank 1
# ------------------------------------------------------------

case_1_result = get_case_result(
    controlled_results_df,
    "AI-ES-001-Q01",
)

assert case_1_result["recall_at_3"] == 1.0
assert case_1_result["recall_at_5"] == 1.0


# ------------------------------------------------------------
# Caso 2 — evidencia relevante en rank 4
# ------------------------------------------------------------

case_2_result = get_case_result(
    controlled_results_df,
    "AI-ES-001-Q03",
)

assert case_2_result["recall_at_3"] == 0.0
assert case_2_result["recall_at_5"] == 1.0


# ------------------------------------------------------------
# Caso 3 — dos chunks relevantes
# ------------------------------------------------------------

case_3_result = get_case_result(
    controlled_results_df,
    "AI-ES-002-Q05",
)

assert case_3_result["recall_at_3"] == 0.5
assert case_3_result["recall_at_5"] == 1.0


# ------------------------------------------------------------
# Caso 4 — no_results
# ------------------------------------------------------------

case_4_result = get_case_result(
    controlled_results_df,
    "BE-ES-001-Q01",
)

assert case_4_result["status"] == "no_results"
assert case_4_result["metric_eligible"] == True
assert case_4_result["recall_at_3"] == 0.0
assert case_4_result["recall_at_5"] == 0.0


# ------------------------------------------------------------
# Caso 5 — error técnico
# ------------------------------------------------------------

case_5_result = get_case_result(
    controlled_results_df,
    "DS-ES-001-Q01",
)

assert case_5_result["status"] == "error"
assert case_5_result["metric_eligible"] == False
assert pd.isna(case_5_result["recall_at_3"])
assert pd.isna(case_5_result["precision_at_5"])

print("✅ Todas las pruebas controladas pasaron correctamente.")

✅ Todas las pruebas controladas pasaron correctamente.


## 9. Resumen de métricas

Se calculan métricas agregadas únicamente sobre casos elegibles.

Los casos con `status = error` se contabilizan como fallos técnicos y no participan en los promedios de calidad del retrieval.

In [41]:
controlled_metric_summary = build_metric_summary(
    controlled_results_df
)

controlled_metric_summary

,metric,mean
0,recall_at_3,0.375000
1,recall_at_5,0.750000
2,precision_at_3,0.166667
3,precision_at_5,0.200000


In [42]:
controlled_status_summary = build_status_summary(
    controlled_results_df
)

controlled_status_summary

,status,count,percentage
0,success,3,60.0
1,no_results,1,20.0
2,error,1,20.0


In [43]:
controlled_category_summary = build_category_summary(
    controlled_results_df
)

controlled_category_summary

,categoria,recall_at_3,recall_at_5,precision_at_3,precision_at_5
0,Backend,0.0,0.0,0.000000,0.000000
1,IA,0.5,1.0,0.222222,0.266667


## 10. Exportación de resultados MOCK

Los siguientes archivos contienen resultados **simulados** utilizados exclusivamente para validar el pipeline de evaluación antes de recibir el retrieval real del equipo de Agentes.

Estos archivos permiten comprobar:

- cálculo de Recall@3 y Recall@5;
- cálculo de Precision@3 y Precision@5;
- manejo de `success`;
- manejo de `no_results`;
- separación de errores técnicos;
- generación de resúmenes globales y por estado.

> **Importante:** estos resultados no representan el rendimiento real del sistema NuevaMente y no deben utilizarse como métricas finales del proyecto.

In [44]:
RESULTS_DIR = EVALUATION_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RETRIEVAL_RESULTS_MOCK_PATH = (
    RESULTS_DIR
    / "retrieval_results_mock_v1.csv"
)

RETRIEVAL_METRICS_MOCK_PATH = (
    RESULTS_DIR
    / "retrieval_metrics_mock_v1.csv"
)

RETRIEVAL_STATUS_MOCK_PATH = (
    RESULTS_DIR
    / "retrieval_status_summary_mock_v1.csv"
)

In [45]:
controlled_results_df.to_csv(
    RETRIEVAL_RESULTS_MOCK_PATH,
    index=False,
    encoding="utf-8-sig",
)

controlled_metric_summary.to_csv(
    RETRIEVAL_METRICS_MOCK_PATH,
    index=False,
    encoding="utf-8-sig",
)

controlled_status_summary.to_csv(
    RETRIEVAL_STATUS_MOCK_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("✅ Archivos MOCK exportados:")
print(RETRIEVAL_RESULTS_MOCK_PATH)
print(RETRIEVAL_METRICS_MOCK_PATH)
print(RETRIEVAL_STATUS_MOCK_PATH)

print(
    "\n⚠️ Estos archivos contienen datos simulados "
    "para validar el pipeline de evaluación."
)

print(
    "⚠️ No representan métricas reales del sistema "
    "ni resultados del equipo de Agentes."
)

✅ Archivos MOCK exportados:
C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\results\retrieval_results_mock_v1.csv
C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\results\retrieval_metrics_mock_v1.csv
C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\results\retrieval_status_summary_mock_v1.csv

⚠️ Estos archivos contienen datos simulados para validar el pipeline de evaluación.
⚠️ No representan métricas reales del sistema ni resultados del equipo de Agentes.


## 11. Preparación para resultados reales de Agentes

El pipeline ya fue validado con datos MOCK.

La siguiente etapa prepara el Notebook 02 para consumir resultados reales del equipo de Agentes manteniendo el mismo contrato `retrieval_contract_v1`.

Los resultados reales deberán:

- respetar `contract_version = 1.0`;
- utilizar `top_k = 5`;
- incluir `case_id`;
- mantener `chunk_id` compatibles con Ground Truth v1;
- utilizar los estados `success`, `no_results` o `error`;
- incluir resultados ordenados por `rank`.

Mientras Agentes no entregue el archivo real, esta sección no genera métricas finales.

In [46]:
INPUT_DIR = EVALUATION_DIR / "input"

INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

AGENTS_RETRIEVAL_PATH = (
    INPUT_DIR
    / "retrieval_results_agentes_v1.json"
)

print("Ruta esperada para Agentes:")
print(AGENTS_RETRIEVAL_PATH)

Ruta esperada para Agentes:
C:\Users\MAMÁ\Downloads\NuevaMente\data\evaluation\input\retrieval_results_agentes_v1.json


In [47]:
if AGENTS_RETRIEVAL_PATH.exists():

    print(
        "✅ Se encontró el archivo real "
        "de retrieval de Agentes."
    )

else:

    print(
        "⏳ Retrieval real de Agentes "
        "todavía no disponible."
    )

    print(
        "El Notebook continuará utilizando "
        "los datos MOCK para validación."
    )

✅ Se encontró el archivo real de retrieval de Agentes.


## 12. Ejecución con resultados reales de Agentes

Esta sección se ejecuta únicamente si existe el archivo:

`data/evaluation/input/retrieval_results_agentes_v1.json`

Si el archivo está disponible:

1. se carga el lote;
2. se valida contra Retrieval Contract v1;
3. se verifica integridad del lote;
4. se evalúan los casos contra Ground Truth v1;
5. se generan métricas reales.

Si el archivo todavía no existe, el notebook finaliza en modo MOCK sin producir métricas finales del sistema.

In [48]:
real_results_df = None
real_report = None

if AGENTS_RETRIEVAL_PATH.exists():

    agents_payloads = load_retrieval_batch(
        AGENTS_RETRIEVAL_PATH
    )

    real_results_df, real_report = (
        evaluate_retrieval_batch(
            agents_payloads,
            ground_truth_df,
        )
    )

    print("✅ Resultados reales cargados y evaluados.")
    print()

    print_batch_report(
        real_report
    )

else:

    print(
        "⏳ No hay resultados reales de Agentes todavía."
    )

    print(
        "El Notebook 02 permanece en modo MOCK."
    )

✅ Resultados reales cargados y evaluados.

Casos esperados: 50
Payloads recibidos: 50
Casos evaluados: 50
Casos faltantes: 0
Casos extra: 0
Case ID duplicados: 0
Errores de procesamiento: 0


In [49]:
real_batch_ready = False

if real_report is not None:

    real_batch_ready = (
        real_report["expected_cases"] == 50
        and real_report["received_payloads"] == 50
        and real_report["evaluated_cases"] == 50
        and len(real_report["missing_case_ids"]) == 0
        and len(real_report["extra_case_ids"]) == 0
        and len(real_report["duplicated_case_ids"]) == 0
        and len(real_report["batch_errors"]) == 0
    )

    if real_batch_ready:
        print(
            "✅ El lote real está completo "
            "y listo para métricas finales."
        )

    else:
        print(
            "⚠️ El lote real todavía no cumple "
            "los criterios de integridad."
        )

else:

    print(
        "ℹ️ No se evaluó integridad real "
        "porque Agentes aún no entregó el lote."
    )

✅ El lote real está completo y listo para métricas finales.


In [50]:
real_metric_summary = None
real_status_summary = None
real_category_summary = None

if real_batch_ready:

    real_metric_summary = build_metric_summary(
        real_results_df
    )

    real_status_summary = build_status_summary(
        real_results_df
    )

    real_category_summary = build_category_summary(
        real_results_df
    )

    print(
        "✅ Resúmenes reales calculados."
    )

else:

    print(
        "⏳ Métricas reales no calculadas todavía."
    )

✅ Resúmenes reales calculados.


In [51]:
if real_batch_ready:

    REAL_RESULTS_PATH = (
        RESULTS_DIR
        / "retrieval_results_gt_v2.csv"
    )

    REAL_METRICS_PATH = (
        RESULTS_DIR
        / "retrieval_metrics_gt_v2.csv"
    )

    REAL_STATUS_PATH = (
        RESULTS_DIR
        / "retrieval_status_summary_gt_v2.csv"
    )

    REAL_CATEGORY_PATH = (
        RESULTS_DIR
        / "retrieval_category_summary_gt_v2.csv"
    )

    real_results_df.to_csv(
        REAL_RESULTS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    real_metric_summary.to_csv(
        REAL_METRICS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    real_status_summary.to_csv(
        REAL_STATUS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    real_category_summary.to_csv(
        REAL_CATEGORY_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        "✅ Resultados reales exportados."
    )

else:

    print(
        "🔒 Exportación real bloqueada "
        "hasta recibir un lote válido de Agentes."
    )

✅ Resultados reales exportados.
